# COMP3314 Machine Learning Challenge – Image Classifier

A starter notebook for the Kaggle image classification competition.  
**Constraints:** No neural networks (CNNs, RNNs, Transformers, etc.), no additional datasets, no pre-trained models.

## Approach
1. Load & explore the dataset
2. Extract handcrafted features: HOG, LBP, colour histograms
3. Reduce dimensionality with PCA
4. Train and compare multiple classical classifiers (SVM, Random Forest, Gradient Boosting, KNN)
5. Select the best model, generate predictions, and create the submission file

## 0. Configuration
Edit the paths and hyper-parameters in this cell before running the notebook.

In [ ]:
import os

# ── Dataset paths ────────────────────────────────────────────────────────────
# Expected layout:
#   data/
#     train/
#       class_a/  img1.jpg  img2.jpg  …
#       class_b/  …
#     test/
#       img1.jpg  img2.jpg  …
#     sample_submission.csv
TRAIN_DIR   = os.path.join('data', 'train')
TEST_DIR    = os.path.join('data', 'test')
SAMPLE_SUB  = os.path.join('data', 'sample_submission.csv')
OUTPUT_SUB  = 'submission.csv'

# ── Image pre-processing ─────────────────────────────────────────────────────
IMG_SIZE    = (64, 64)   # resize all images to this (H, W)

# ── HOG parameters ───────────────────────────────────────────────────────────
HOG_ORIENTATIONS   = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)

# ── LBP parameters ───────────────────────────────────────────────────────────
LBP_RADIUS  = 3
LBP_POINTS  = 8 * LBP_RADIUS

# ── Colour histogram parameters ───────────────────────────────────────────────
HIST_BINS   = 32

# ── PCA ──────────────────────────────────────────────────────────────────────
PCA_VARIANCE = 0.95   # keep enough components to explain 95% of variance

# ── Cross-validation ─────────────────────────────────────────────────────────
CV_FOLDS    = 5
RANDOM_SEED = 42

# ── CI mode (set by the workflow; limits data so the notebook runs fast) ──────
CI_MODE     = os.getenv('CI_MODE', 'false').lower() == 'true'
CI_MAX_IMGS = 200   # images per class used in CI

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

from PIL import Image
from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

import joblib

sns.set_theme(style='whitegrid', palette='tab10')
print('All imports successful.')

## 2. Data Loading

In [ ]:
def load_images_from_dir(directory, label=None, max_per_class=None):
    """Recursively load images from *directory*.

    If *label* is None the function expects class sub-folders (train layout).
    If *label* is provided all images are assigned that label (test layout).

    Returns
    -------
    images : list of np.ndarray  (H, W, 3) uint8
    labels : list of str
    filenames : list of str
    """
    images, labels, filenames = [], [], []
    root = Path(directory)

    if label is None:
        # Train layout: sub-folder name == class label
        class_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
        for class_dir in class_dirs:
            class_label = class_dir.name
            img_paths = sorted(
                p for p in class_dir.iterdir()
                if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
            )
            if max_per_class is not None:
                img_paths = img_paths[:max_per_class]
            for p in img_paths:
                try:
                    img = Image.open(p).convert('RGB').resize(IMG_SIZE)
                    images.append(np.array(img))
                    labels.append(class_label)
                    filenames.append(p.name)
                except Exception as e:
                    print(f'  Warning: could not load {p}: {e}')
    else:
        # Test layout: flat directory
        img_paths = sorted(
            p for p in root.iterdir()
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
        )
        for p in img_paths:
            try:
                img = Image.open(p).convert('RGB').resize(IMG_SIZE)
                images.append(np.array(img))
                labels.append(label)
                filenames.append(p.name)
            except Exception as e:
                print(f'  Warning: could not load {p}: {e}')

    return images, labels, filenames

In [ ]:
max_imgs = CI_MAX_IMGS if CI_MODE else None

print('Loading training images…')
train_imgs, train_labels, train_files = load_images_from_dir(
    TRAIN_DIR, max_per_class=max_imgs
)

print('Loading test images…')
test_imgs, test_labels, test_files = load_images_from_dir(
    TEST_DIR, label='unknown'
)

print(f'\nTraining samples : {len(train_imgs)}')
print(f'Test samples     : {len(test_imgs)}')
print(f'Classes          : {sorted(set(train_labels))}')

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
label_counts = pd.Series(train_labels).value_counts().sort_index()
plt.figure(figsize=(max(8, len(label_counts) * 0.6), 4))
label_counts.plot(kind='bar', edgecolor='white')
plt.title('Training set – class distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(label_counts.to_string())

In [ ]:
# Sample images from each class
classes = sorted(set(train_labels))
n_cols = min(5, len(classes))
n_rows = (len(classes) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
axes = np.array(axes).flatten()

for ax, cls in zip(axes, classes):
    idx = train_labels.index(cls)
    ax.imshow(train_imgs[idx])
    ax.set_title(cls, fontsize=9)
    ax.axis('off')

for ax in axes[len(classes):]:
    ax.axis('off')

plt.suptitle('Sample images per class', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 4. Feature Extraction

Three complementary feature families are extracted and concatenated:

| Feature | Captures |
|---------|----------|
| **HOG** | Shape and edge structure |
| **LBP** | Texture information |
| **Colour histogram** | Global colour distribution |

In [ ]:
def extract_hog(img):
    """HOG features from a greyscale version of *img*."""
    gray = rgb2gray(img)
    return hog(
        gray,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm='L2-Hys',
        feature_vector=True,
    )


def extract_lbp(img):
    """LBP histogram features."""
    gray = rgb2gray(img)
    lbp = local_binary_pattern(
        gray, P=LBP_POINTS, R=LBP_RADIUS, method='uniform'
    )
    n_bins = LBP_POINTS + 2
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins,
                            range=(0, n_bins), density=True)
    return hist


def extract_colour_hist(img):
    """Per-channel colour histograms (R, G, B) concatenated."""
    hists = []
    for ch in range(3):
        h, _ = np.histogram(img[:, :, ch], bins=HIST_BINS,
                             range=(0, 256), density=True)
        hists.append(h)
    return np.concatenate(hists)


def extract_features(images):
    """Extract and concatenate all feature families for a list of images."""
    feat_list = []
    for img in tqdm(images, desc='Extracting features'):
        hog_feat   = extract_hog(img)
        lbp_feat   = extract_lbp(img)
        chist_feat = extract_colour_hist(img)
        feat_list.append(np.concatenate([hog_feat, lbp_feat, chist_feat]))
    return np.array(feat_list, dtype=np.float32)


print('Extracting training features…')
X_train_raw = extract_features(train_imgs)
print('Extracting test features…')
X_test_raw  = extract_features(test_imgs)

print(f'\nFeature vector size: {X_train_raw.shape[1]}')

## 5. Label Encoding

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(train_labels)
print('Classes:', le.classes_)

## 6. Pre-processing Pipeline (Scaler + PCA)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_raw)

pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_SEED)
X_train = pca.fit_transform(X_scaled)
X_test  = pca.transform(scaler.transform(X_test_raw))

print(f'PCA reduced {X_train_raw.shape[1]} → {X_train.shape[1]} features '
      f'(explaining {PCA_VARIANCE*100:.0f}% variance)')

## 7. Model Training and Cross-Validation

Four classic classifiers are evaluated with stratified k-fold CV.  
The best model (by mean accuracy) is then retrained on the full training set.

In [ ]:
models = {
    'SVM (RBF)': SVC(
        kernel='rbf', C=10, gamma='scale',
        decision_function_shape='ovr', random_state=RANDOM_SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=None,
        n_jobs=-1, random_state=RANDOM_SEED
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1,
        max_depth=4, random_state=RANDOM_SEED
    ),
    'KNN (k=7)': KNeighborsClassifier(
        n_neighbors=7, weights='distance', n_jobs=-1
    ),
}

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_results = {}
for name, clf in models.items():
    print(f'Cross-validating {name}…', end=' ', flush=True)
    scores = cross_val_score(
        clf, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1
    )
    cv_results[name] = scores
    print(f'mean={scores.mean():.4f}  std={scores.std():.4f}')

In [ ]:
# Plot CV results
cv_df = pd.DataFrame(cv_results).melt(var_name='Model', value_name='Accuracy')
plt.figure(figsize=(9, 5))
sns.boxplot(data=cv_df, x='Model', y='Accuracy')
sns.stripplot(data=cv_df, x='Model', y='Accuracy',
              color='black', size=4, jitter=True)
plt.title(f'{CV_FOLDS}-Fold CV Accuracy')
plt.xticks(rotation=15, ha='right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# Summary table
summary = pd.DataFrame({
    name: {'mean': s.mean(), 'std': s.std(), 'max': s.max()}
    for name, s in cv_results.items()
}).T.sort_values('mean', ascending=False)
print(summary.to_string(float_format='{:.4f}'.format))

In [ ]:
# Select the best model
best_model_name = summary.index[0]
best_model = models[best_model_name]
print(f'Best model: {best_model_name}  (mean CV accuracy: {summary.loc[best_model_name, "mean"]:.4f})')

## 8. Final Training on Full Training Set

In [ ]:
print(f'Training {best_model_name} on full training set…')
best_model.fit(X_train, y_train)
train_acc = accuracy_score(y_train, best_model.predict(X_train))
print(f'Training accuracy: {train_acc:.4f}')

## 9. Evaluation on Training Set (Sanity Check)

In [ ]:
y_pred_train = best_model.predict(X_train)
print(classification_report(y_train, y_pred_train, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_train, y_pred_train)
plt.figure(figsize=(max(6, len(le.classes_) * 0.9),
                     max(5, len(le.classes_) * 0.8)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix – Training Set')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

## 10. Predict on Test Set & Create Submission

In [ ]:
y_test_pred   = best_model.predict(X_test)
test_labels_decoded = le.inverse_transform(y_test_pred)

submission = pd.DataFrame({
    'filename': test_files,
    'label':    test_labels_decoded,
})

# Align to sample submission column order if available
if Path(SAMPLE_SUB).exists():
    sample = pd.read_csv(SAMPLE_SUB)
    submission = sample[['filename']].merge(submission, on='filename', how='left')

submission.to_csv(OUTPUT_SUB, index=False)
print(f'Submission saved to {OUTPUT_SUB}')
submission.head(10)

## 11. Save Best Model (optional)

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(
    {'model': best_model, 'scaler': scaler, 'pca': pca, 'label_encoder': le},
    os.path.join('models', 'best_model.joblib')
)
print('Model saved to models/best_model.joblib')